# CIFAR-10 Baseline Experiment

Train the baseline CNN on CIFAR-10 and log metrics.

In [ ]:
import sys
from pathlib import Path
import time

candidate_roots = [
    Path('/content/ouroboros'),
]
project_root = next((p for p in candidate_roots if p.exists()), None)
if project_root is None:
    raise FileNotFoundError('Project root not found. Update candidate_roots.')
sys.path.insert(0, str(project_root))

import torch
from torch import nn
from torch.optim import Adam

from src.data_loaders import get_cifar10_loaders
from src.metrics import MetricsLogger, compute_system_metrics, plot_learning_curves, reset_cuda_peak_memory
from src.models import CNN3Layer
from src.trainer import train_epoch, validate_epoch, save_checkpoint
from src.utils import get_device, set_seed, ensure_dirs

set_seed(42)
device = get_device()
ensure_dirs('results', 'results/figures', 'checkpoints')

lr = 1e-3
batch_size = 128
epochs = 5

train_loader, val_loader = get_cifar10_loaders(batch_size, 2, 'assets')
model = CNN3Layer(num_classes=10, in_channels=3).to(device)
optimizer = Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

logger = MetricsLogger(run_metadata={'dataset': 'CIFAR-10', 'epochs': epochs})

for epoch in range(1, epochs + 1):
    reset_cuda_peak_memory()
    start = time.perf_counter()
    train_metrics = train_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        amp_enabled=(device.type == 'cuda'),
        use_compile=hasattr(torch, 'compile'),
        collect_grad_stats=True,
    )
    val_metrics = validate_epoch(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )
    system_metrics = compute_system_metrics(
        total_samples=len(train_loader) * batch_size,
        start_time=start,
        end_time=time.perf_counter(),
        device=device,
    )
    logger.log_epoch(
        epoch=epoch,
        train={k: v for k, v in train_metrics.items() if k != 'gradients'},
        validation=val_metrics,
        gradients=train_metrics.get('gradients'),
        system=system_metrics,
    )
    print('Epoch', epoch, 'train', train_metrics, 'val', val_metrics)

metrics_path = Path('results') / 'cifar10_baseline_metrics.json'
logger.to_json(metrics_path)
plot_learning_curves(metrics_path, output_dir='results/figures', prefix='cifar10_baseline')

checkpoint_path = Path('checkpoints') / 'cifar10_baseline.pth'
final_metrics = {
    'train_loss': logger.epoch_metrics[-1]['train'].get('loss', 0.0),
    'train_accuracy': logger.epoch_metrics[-1]['train'].get('accuracy', 0.0),
    'val_loss': logger.epoch_metrics[-1]['validation'].get('loss', 0.0),
    'val_accuracy': logger.epoch_metrics[-1]['validation'].get('accuracy', 0.0),
}
save_checkpoint(str(checkpoint_path), model, optimizer, epochs, final_metrics)
print('Saved metrics to', metrics_path)
print('Saved checkpoint to', checkpoint_path)

100%|██████████| 170M/170M [00:08<00:00, 20.1MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1 train {'loss': 1.572133201513535, 'accuracy': 0.4345352564102564, 'gradients': {'total_l2_norm': 4.247225084999417, 'per_layer_l2_norms': {'_orig_mod.conv1.weight': 3.9865856170654297, '_orig_mod.conv1.bias': 5.889994281460531e-05, '_orig_mod.bn1.weight': 0.12440027296543121, '_orig_mod.bn1.bias': 0.08182326704263687, '_orig_mod.conv2.weight': 1.3281583786010742, '_orig_mod.conv2.bias': 5.517595695891941e-07, '_orig_mod.bn2.weight': 0.06265740841627121, '_orig_mod.bn2.bias': 0.056710854172706604, '_orig_mod.conv3.weight': 0.34320417046546936, '_orig_mod.conv3.bias': 4.287326476060116e-07, '_orig_mod.bn3.weight': 0.04617702215909958, '_orig_mod.bn3.bias': 0.05373166874051094, '_orig_mod.fc.weight': 0.472693532705307, '_orig_mod.fc.bias': 0.08056750893592834}, 'zero_grad_parameters': 0}} val {'loss': 1.5125246850967407, 'accuracy': 0.4462}
Epoch 2 train {'loss': 1.2722835120482323, 'accuracy': 0.5471354166666667, 'gradients': {'total_l2_norm': 5.281600376171307, 'per_layer_l2_nor

# Conclusion

The model demonstrates stable convergence. Training loss decreases monotonically (1.57 -> 1.06) with corresponding accuracy gains (43% -> 62.6%), indicating effective parameter optimization. Validation metrics track training trends, suggesting no significant overfitting within 5 epochs.

Gradient norms remain bounded (~4-5 L2), confirming numerically stable backpropagation and healthy signal propagation across layers. The temporary validation fluctuation (Epoch 3) appears stochastic rather than structural.

Overall, results indicate a well-conditioned baseline CNN with expected CIFAR-10 early-stage performance. Further gains likely depend on longer training, augmentation, or deeper architectures.